## **CUDA Runtime**

- Copies the data from the host to the device

- Load GPU kernel and execute it

- Copy the result from the device to the host

<hr>

## **`__global__`, `__device__`, and `__host__`**

Remember how we talked about `nvcc` acting as a "Compiler Driver" that cuts your code in half, sending the CPU code to `g++` and the GPU code to the PTX pipeline?

Because both your CPU code and your GPU code live inside the exact same `.cu` file, `nvcc` needs a way to know **which function belongs to which processor**. 

To solve this, CUDA introduces **Execution Space Specifiers**: `__host__`, `__global__`, and `__device__`. Think of these as nametags you put on your functions. They tell the compiler two very specific things:
1. **Where does this function run?** (CPU or GPU)
2. **Who is allowed to call this function?** (CPU or GPU)

Here is the in-depth breakdown of each, why they exist, and how they interact.

---

### 1. `__host__` (The Standard CPU Function)

* **Executes on:** The Host (CPU)

* **Callable from:** The Host (CPU) ONLY, we can't call from the GPU Kernel

**What it is:**

This is just a normal, everyday C++ function. In fact, if you write a function and don't put *any* tag on it, the compiler automatically assumes it is `__host__`. 

**Why it exists:**

You almost never have to type `__host__` explicitly. It exists mostly so you can combine it with `__device__` (which I will show you in the "Secret Combo" below!).

```cpp
// You don't need to type __host__, but this is exactly what it means!
__host__ void printHelloFromCPU() {
    printf("I am running on the AMD Ryzen CPU!\n");
}

int main() {
    printHelloFromCPU(); // CPU calls a CPU function. Perfectly fine.
    return 0;
}
```

---

### 2. `__global__` (The Bridge / The Kernel)

* **Executes on:** The Device (GPU)

* **Callable from:** The Host (CPU)*, but with a special syntax: `<<<blocks, threads>>>`. Executed by thousands of GPU threads in parallel.

**What it is:**

The `__global__` tag is the most important word in CUDA. It defines a **Kernel**. This is the main door, or the "bridge," between your CPU and your GPU. It allows your CPU to order the GPU to start doing work.

**Strict Rules for `__global__`:**

1. It **must** return `void`. A GPU kernel cannot use `return 5;` back to the CPU, because the CPU doesn't wait for the GPU to finish (it is asynchronous). To get data back, you must write it to a pointer in VRAM and use `cudaMemcpy`.

2. Whenever the CPU calls a `__global__` function, it **must** use the execution configuration syntax: `<<<blocks, threads>>>`.

```cpp
// Executes on the RTX 3060, but is triggered by the Ryzen CPU
__global__ void addMathKernel(int* d_array) {
    int tid = threadIdx.x;
    d_array[tid] = d_array[tid] + 10;
}

int main() {
    // The CPU stands on the edge of the bridge and shouts the command over to the GPU
    addMathKernel<<<1, 256>>>(d_array); 
    return 0;
}
```

*(Fun Advanced Fact: Since CUDA 5.0, NVIDIA introduced "Dynamic Parallelism", which actually allows a `__global__` function to call another `__global__` function right from the GPU, but as a beginner, just think of it as the CPU-to-GPU bridge!)*

---

### 3. `__device__` (The GPU Helper Function)

* **Executes on:** The Device (GPU)

* **Callable from:** The Device (GPU) ONLY

**What it is:**

As your GPU programs get extremely complex, you don't want to write a 1,000-line `__global__` kernel. You want to break your code up into smaller, modular functions just like you do in normal C++. 

A `__device__` function is a helper function that lives entirely inside the GPU. 

* The CPU cannot see it.

* The CPU cannot call it.

* It can only be called by a `__global__` kernel, or by another `__device__` function.

**Unlike `__global__`, a `__device__` function CAN return values!**

```cpp
// 1. The Helper Function (GPU Only)
__device__ int squareNumber(int a) {
    return a * a; // It can return values!
}

// 2. The Main Kernel (GPU execution, CPU trigger)
__global__ void processArray(int* d_array) {
    int tid = threadIdx.x;
    
    // The GPU kernel calls its own internal GPU helper function
    d_array[tid] = squareNumber(d_array[tid]); 
}

int main() {
    processArray<<<1, 256>>>(d_array);
    
    // squareNumber(5); // COMPILER ERROR! The CPU cannot reach a __device__ function.
    return 0;
}
```

---

### 4. The Secret Combo: `__host__ __device__` 

Sometimes, you write a fantastic math function (like calculating the distance between two 3D points). 

You want to use it in your CPU code to test a single point, but you also want to use it inside your GPU kernel to process a million points.

If you tag it as `__host__`, the GPU can't use it.
If you tag it as `__device__`, the CPU can't use it.
If you don't tag it at all, `nvcc` assumes it's `__host__`.

**The Solution:** You give it **both tags**.

```cpp
__host__ __device__ float getDistance(float x, float y) {
    return sqrt((x * x) + (y * y));
}
```
**What happens under the hood:**

When `nvcc` sees both tags, it literally compiles **two entirely separate versions** of the exact same function. It builds an x86 machine-code version for your Ryzen CPU, and a `PTX (Parallel Thread Execution)/SASS` version for your RTX 3060. Now, both your `main()` function and your `__global__` kernels can call `getDistance()` flawlessly!

> PTX (Parallel Thread Execution). It is a low-level, hardware-agnostic Intermediate Representation (IR) and virtual Instruction Set Architecture (ISA) used for writing and optimizing GPU code. It is an intermediate language that sits between high-level CUDA C++ code and the final machine code (SASS) that runs on NVIDIA GPUs. It has `.ptx` files, which are human-readable assembly-like code that can be further compiled into SASS (the actual machine code for the GPU).

> SASS (Streaming Assembler). It is the final machine code that runs directly on NVIDIA GPUs. It is generated from PTX code by the NVIDIA driver at runtime. `SASS` is specific to the architecture of the GPU (e.g., Volta, Turing, Ampere) and is not human-readable. It is optimized for performance and efficiency on the target GPU. The `PTX` is compiled into `SASS` (`.cubin` files) by the NVIDIA driver when the program is executed.

---

### Putting it all together (The Master Example)

Here is a complete, single program that shows exactly how all three interact in a real CUDA codebase. Read the comments to see who is calling who.

```cpp
#include <iostream>
#include <stdio.h>

// ---------------------------------------------------------
// 1. HOST-ONLY: Normal C++ function
// ---------------------------------------------------------
__host__ void printIntro() {
    printf("Starting the program on the CPU...\n");
}

// ---------------------------------------------------------
// 2. DEVICE-ONLY: Helper function for the GPU
// ---------------------------------------------------------
__device__ int doubleTheValue(int val) {
    return val * 2;
}

// ---------------------------------------------------------
// 3. HOST & DEVICE: Can be called by anyone!
// ---------------------------------------------------------
__host__ __device__ int addFive(int val) {
    return val + 5;
}

// ---------------------------------------------------------
// 4. GLOBAL: The Bridge. Runs on GPU, called by CPU.
// ---------------------------------------------------------
__global__ void myKernel(int* d_data) {
    int tid = threadIdx.x;
    
    // Kernel calls a __device__ function
    int doubled = doubleTheValue(d_data[tid]); 
    
    // Kernel calls a __host__ __device__ function
    d_data[tid] = addFive(doubled); 
}

// ---------------------------------------------------------
// 5. MAIN: The starting point on the CPU
// ---------------------------------------------------------
int main() {
    // 1. CPU calls a __host__ function
    printIntro(); 

    // 2. CPU calls a __host__ __device__ function (Just to test it!)
    int cpu_test = addFive(10); 
    printf("CPU tested addFive(10) and got: %d\n", cpu_test);

    // ... (Imagine cudaMalloc and cudaMemcpy happen here) ...
    int* d_data;

    // 3. CPU calls a __global__ function to wake up the GPU
    // myKernel<<<1, 32>>>(d_data);

    // 4. CPU waits for GPU to finish
    // cudaDeviceSynchronize();

    return 0;
}
```

<hr>

## **Blocks and Threads**

**What is `int tid = (blockIdx.x * blockDim.x) + threadIdx.x;`?**

This is the single most confusing line of code for every beginner learning CUDA. But once you understand the real-world logic behind it, you will never forget it.

To understand this formula, you have to understand how the GPU spawns threads. 

The GPU does not just spawn 1,000 threads in a straight line. It groups them into **Blocks**. 

### The Apartment Building Analogy
Imagine you are managing a neighborhood of **Apartment Buildings**.
* Every building is identical.
* Every building has exactly **10 rooms**.

Now, imagine I hire you as a delivery driver, and I ask you: *"Deliver this package to Room Number 24 on the street."*

But there is a problem: The doors inside the buildings only have numbers from `0` to `9`. There is no door labeled "24". 

How do you find Global Room 24? You use math:
1. You know every building has 10 rooms.
2. So, Building 0 has rooms 0-9.
3. Building 1 has rooms 10-19.
4. Building 2 has rooms 20-29.
5. Therefore, you walk to **Building 2**, and you go to local **door 4** inside that building.

### The CUDA Translation
That exact math is what `int tid = (blockIdx.x * blockDim.x) + threadIdx.x;` is doing, just in reverse. A single GPU thread wakes up inside a building and says: *"Wait, what is my Global ID?"*

Let's translate the CUDA keywords to our analogy:

* **`blockDim.x` (Rooms per building):** The size of the block. How many threads are in every single block? (Let's say **10**).
* **`blockIdx.x` (Building ID):** Which block is this thread currently standing in? (Let's say Block **2**).
* **`threadIdx.x` (Local Door ID):** Which specific thread is it inside this block? (Let's say Thread **4**).

### Let's do the math step-by-step:

**Step 1: `(blockIdx.x * blockDim.x)`**
* `(2 * 10) = 20`. 
* What does this 20 mean? It means *"Because I am in Block 2, I know there are 2 full blocks (Block 0 and Block 1) that come before me. Since each has 10 threads, I must skip the first 20 threads."*
* This gives you the **starting point** of your block.

**Step 2: `+ threadIdx.x`**
* `20 + 4 = 24`.
* What does this mean? *"I start at thread 20, and I am the 4th person inside my block. Therefore, my absolute, global ID across the entire GPU is 24."*

### A Visual Representation
Let's look at a smaller example. Imagine we launch a kernel with **3 Blocks**, and **4 Threads per Block**.

```text
       BLOCK 0                 BLOCK 1                 BLOCK 2
[ 0 ][ 1 ][ 2 ][ 3 ]    [ 0 ][ 1 ][ 2 ][ 3 ]    [ 0 ][ 1 ][ 2 ][ 3 ]  <-- threadIdx.x (Local ID)
```
Notice how every block restarts counting from 0? If a thread just used `threadIdx.x` to grab a number from an array, Block 0, Block 1, and Block 2 would all try to grab the 0th item at the same time! We don't want that.

So, the thread calculates its Global ID using the formula:
* The thread at **Block 1, Local ID 2** calculates: `(1 * 4) + 2 = 6`.
* The thread at **Block 2, Local ID 3** calculates: `(2 * 4) + 3 = 11`.

```text
       BLOCK 0                 BLOCK 1                 BLOCK 2
[ 0 ][ 1 ][ 2 ][ 3 ]    [ 4 ][ 5 ][ 6 ][ 7 ]    [ 8 ][ 9 ][10 ][11 ]  <-- tid (Global ID)
```

### Why do we need this global `tid`?
Because your arrays in VRAM (`d_data`) are flat, 1D arrays (e.g., `[0, 1, 2, ..., 1000000]`). 

By calculating `tid`, the GPU guarantees that out of 1 million threads, **every single thread gets a perfectly unique integer**, ensuring that every thread modifies exactly one unique slot in your array!

<hr>

## **Memory Management**

We just cannot directly use our CPU's RAM from the GPU. The GPU has its own separate memory called VRAM. At first everything is in the `RAM` of the CPU, and if we want to use it on the GPU, we have to explicitly copy it over to the VRAM. The GPU will use VRAM to do all its processing, and then if we want to get the results back to the CPU, we have to copy it back from VRAM to RAM.

### 1. `cudaMalloc`

**What it does:**

It reserves a contiguous block of memory inside the **Global Memory (VRAM)** of your RTX 3060. 

**How the syntax works:**

```cpp
float *d_a; 
cudaMalloc(&d_a, N*N*sizeof(float));
```

* **`N*N`**: This instantly tells us we are dealing with a flattened 2D grid! (Like an image with `N` width and `N` height).

* **The Pointer Paradox (`&d_a`)**: Notice that `float *d_a;` is created on the CPU. It is a CPU variable. But we want it to hold a GPU memory address. We pass `&d_a` (the address of the pointer) to `cudaMalloc` so the NVIDIA driver can physically modify our CPU pointer and inject the brand-new GPU memory address into it. 

**Why it is called "Global Memory":**

The VRAM on your graphics card is called "Global" because **everyone can see it**. The CPU can see it (via `cudaMemcpy`), and every single one of the thousands of GPU threads can read and write to it simultaneously.

---

### 2. `cudaMemcpy`

**What it does:**

It physically moves bytes of data across the PCIe bus on your motherboard. 

**Why we need directions:**

The hardware needs to know which way the traffic is flowing to optimize the DMA (Direct Memory Access) controllers. 

```bash
# Syntax
cudaMemcpy( Destination, Source, Size_in_Bytes, Direction );
cudaMemcpy(d_image_in, h_image_in.data(), bytes, cudaMemcpyHostToDevice);
```

* **`cudaMemcpyHostToDevice` (CPU to GPU):** Used at the start of your program. You load a file from your SSD into CPU RAM, and then ship it to the GPU to be processed.

* **`cudaMemcpyDeviceToHost` (GPU to CPU):** Used at the end of your program. The GPU cannot save files to your SSD. It must ship the finished math back to the CPU so your C++ program can save the file.

* **`cudaMemcpyDeviceToDevice` (GPU to GPU):** Used purely inside the VRAM. **Why do this?** Imagine you are blurring an image. You need to read the original pixels to calculate the blur, but write the blurred pixels to a new location so you don't corrupt the original data mid-calculation. Copying from one VRAM location to another VRAM location operates at roughly **300+ GB/s**, whereas copying over the PCIe bus is only about **16 GB/s**.

---

### 3. `cudaFree`

**What it does:**

It releases the VRAM back to the NVIDIA driver.

**Why it is critical:**

Your RTX 3060 Laptop GPU has exactly **6 GB of VRAM**. 

If you process a video frame that takes 100 MB of VRAM, and you forget to call `cudaFree` at the end of your loop, your GPU will "leak" 100 MB per frame.

At 60 frames per second, **you will completely fill and crash your graphics card in exactly 1 second.** `cudaFree` is non-negotiable!

---

### The Real-World Example: 4K Image Processing

Let’s step out of abstract "Vector A and Vector B" and look at a real-world scenario. 
Imagine we are writing a Photoshop-style application. We want to take a massive 4K high-dynamic-range (HDR) image and apply a Brightness Filter to it.

HDR image pixels are stored as `float` values (decimals). 
A 4K image is roughly `4000 x 4000` pixels (`N * N`).

Here is the exact memory management pipeline to process that image.

```cpp
#include <iostream>
#include <vector>

int main() {
    // 1. Define our image size
    int N = 4000; // 4000 width x 4000 height
    size_t num_pixels = N * N; // 16,000,000 pixels
    size_t bytes = num_pixels * sizeof(float); // ~64 Megabytes of data

    // ==========================================
    // CPU PHASE: Load the Image
    // ==========================================
    // Create the image array in CPU RAM
    std::vector<float> h_image_in(num_pixels, 0.5f); // Pretend we loaded a gray image
    std::vector<float> h_image_out(num_pixels);      // Empty array to hold the result

    // ==========================================
    // GPU LOGISTICS PHASE 1: Allocate VRAM
    // ==========================================
    float *d_image_in, *d_image_out;

    // We need 64 MB for the original image, and 64 MB for the new brightened image
    cudaMalloc(&d_image_in, bytes);
    cudaMalloc(&d_image_out, bytes);

    // ==========================================
    // GPU LOGISTICS PHASE 2: Ship the Data
    // ==========================================
    std::cout << "Shipping 64MB Image to GPU...\n";
    // Send the raw image from Host (CPU) to Device (GPU)
    cudaMemcpy(d_image_in, h_image_in.data(), bytes, cudaMemcpyHostToDevice);

    // ==========================================
    // COMPUTE PHASE: The Magic Happens
    // ==========================================
    // (Imagine we launch a kernel here that adds +0.2f brightness to every pixel)
    // brightenImageKernel<<<Blocks, Threads>>>(d_image_in, d_image_out, num_pixels);
    
    // Here is a Device-to-Device example! 
    // Let's pretend the kernel failed, and we just want to copy the original 
    // image straight into the output buffer entirely within the GPU VRAM:
    cudaMemcpy(d_image_out, d_image_in, bytes, cudaMemcpyDeviceToDevice);

    // ==========================================
    // GPU LOGISTICS PHASE 3: Retrieve the Product
    // ==========================================
    std::cout << "Retrieving finished Image from GPU...\n";
    // Send the finished, brightened image from Device (GPU) back to Host (CPU)
    cudaMemcpy(h_image_out.data(), d_image_out, bytes, cudaMemcpyDeviceToHost);

    // ==========================================
    // GPU LOGISTICS PHASE 4: Clean Up
    // ==========================================
    // We saved our result to the CPU, so the GPU doesn't need the memory anymore.
    cudaFree(d_image_in);
    cudaFree(d_image_out);

    std::cout << "VRAM Freed. Image ready to save to hard drive!\n";

    return 0;
}
```

<hr>

## **CUDA Execution Hierarchy**

A CUDA kernel is a function that runs on the `GPU`, when we launch it from the `CPU` we decide: 

- How many `Blocks` we want to spawn (The number of rooms in the factory)

- How many `Threads` we want to spawn inside each block (The number of workers in each room)


```c++

kernel<<<3, 4>>>(...);

```

This means: 

- 3 blocks 

- 4 threads per block
 
So total threads = 3 * 4 = 12 threads.

We can picture it like this: 

```bash

BLOCK 0                 BLOCK 1                 BLOCK 2
[ 0 ][ 1 ][ 2 ][ 3 ]    [ 4 ][ 5 ][ 6 ][ 7 ]    [ 8 ][ 9 ][10 ][11 ]
```

Inside CUDA, every thread has: 

- A unique **Thread ID** (threadIdx.x) inside its block (local ID)

- A unique **Block ID** (blockIdx.x) inside the grid (local ID)

---

### Level 1: The Thread (The Individual Worker)

* **Execution:** A `Thread` is one single factory worker. 

* **Memory (Local Memory/Registers):** The worker's own pockets and toolbelt.

**In Depth:**

When you launch a GPU program, the GPU hires millions of workers (threads). Each worker is assigned one tiny, specific job (e.g., "Paint pixel number 5"). 

While working, the worker needs to do some quick math. They write this math down on a notepad in their pocket. This is called **Local Memory (Registers)**. It is insanely fast, but **private**. 

The worker next to them cannot look inside their pocket.

A `Thread` has: 

- It's own `Registers` (Local Memory)

- It's own `Private Local Memory` (The notepad in their pocket)

- Access to `Shared Memory` of it's `Block` (The whiteboard in the room)

- Access to `Global Memory` (The warehouse outside the factory), shared by all threads but very slow to access.

### Level 2: The Block (The Team)

* **Execution:** A Block is a team of workers placed inside one specific locked room.

* **Memory (Shared Memory):** A whiteboard placed in the middle of that room.

**In Depth:**

The GPU doesn't just put 1 million workers in an open field. It organizes them into **Blocks** (Teams). A Block can have a maximum of `1,024` threads. 

**Why group them? Why not just individual workers?**

Because sometimes workers need to collaborate. If you are blurring an image, Worker A needs to know what color Worker B just painted. 

To do this, the GPU gives the team a **Shared Memory** whiteboard (`__shared__` in code). 

* Any worker in the room can write to the whiteboard.

* Any worker in the room can read the whiteboard.

* It is lightning-fast because the whiteboard is physically inside the room with them.

* However, workers in Block 1 **cannot** see the whiteboard in Block 2. The doors are locked.

- Threads in the same block can `Synchronize` with `__syncthreads()` to make sure all workers have finished writing to the whiteboard before anyone reads it.

### Level 3: The Grid (The Entire Factory)

* **Execution:** The Grid is the entire factory building that contains all the rooms (Blocks).

* **Memory (Global Memory):** The giant VRAM warehouse outside the factory.

**In Depth:**

The Grid is the total sum of your kernel launch. If you have 1 million elements to process, you do not want 1 thread doing all the work, instead: 

- Split the work into blocks 

- Split each block into threads

- Let many threads work in parallel to finish the job faster.

This is what makes `CUDA` so powerful. You can have **millions of workers** working in parallel, each doing a tiny piece of the job, and finish in a fraction of the time it would take a single CPU thread.

And,

Workers in Room A cannot see the whiteboard in Room B, what happens if they *need* to share data across the entire factory? 

They have to walk outside to the giant **Global Memory Warehouse** (Your 6GB of RTX 3060 VRAM, allocated via `cudaMalloc`). 

* Everyone in the factory can see the warehouse.

* But walking to the warehouse takes a lot of time (High latency).

---

### **Thread IDs in Detail**

CUDA gives built in variables to identify each `Thread` and `Block`.

For `1D` kernels, the common ones are: 

- `threadIdx.x` - The thread's unique ID inside its block (local ID)

- `blockIdx.x` - The block's unique ID inside the grid (local ID)

- `blockDim.x` - The total number of threads in a block (size of the block)

- `gridDim.x` - The total number of blocks in the grid (size of the grid)

**Example:**

```cpp
kernel<<<3, 4>>>(...);
```

Then, 

- `gridDim.x = 3` (3 blocks in the grid)

- `blockDim.x = 4` (4 threads in each block)

Block 0:

- `threadIdx.x = 0, 1, 2, 3` (local thread IDs)

- `blockIdx.x = 0` (local block ID)

Block 1:

- `threadIdx.x = 0, 1, 2, 3` (local thread IDs)

- `blockIdx.x = 1` (local block ID)

Block 2:

- `threadIdx.x = 0, 1, 2, 3` (local thread IDs)

- `blockIdx.x = 2` (local block ID)

### **Global Thread ID Calculation**

If we want one unique ID for every thread across the entire grid, we can calculate it as:

```cpp
int tid = (blockIdx.x * blockDim.x) + threadIdx.x;
``` 

For example, in the above kernel launch:

- Block 0, Thread 0: `tid = (0 * 4) + 0 = 0`
- Block 0, Thread 1: `tid = (0 * 4) + 1 = 1`
- Block 0, Thread 2: `tid = (0 * 4) + 2 = 2`
- Block 0, Thread 3: `tid = (0 * 4) + 3 = 3`

- Block 1, Thread 0: `tid = (1 * 4) + 0 = 4`
- Block 1, Thread 1: `tid = (1 * 4) + 1 = 5`
- Block 1, Thread 2: `tid = (1 * 4) + 2 = 6`
- Block 1, Thread 3: `tid = (1 * 4) + 3 = 7`

- Block 2, Thread 0: `tid = (2 * 4) + 0 = 8`
- Block 2, Thread 1: `tid = (2 * 4) + 1 = 9`
- Block 2, Thread_2: `tid = (2 * 4) + 2 = 10`
- Block_2, Thread_3: `tid = (2 * 4) + 3 = 11`

---


### **Simple Example**

```cpp
kernel<<<3,4>>>();
```

This means:

* **3 blocks**

* **4 threads per block**

* total = **12 threads**

We’ll build a simple vector addition example.

---

# Problem

Suppose:

```text
A = [1,2,3,4,5,6,7,8,9,10,11,12]

B = [10,20,30,40,50,60,70,80,90,100,110,120]
```

We want:

```text
C = A + B
```

Result:

```text
C = [11,22,33,44,55,66,77,88,99,110,121,132]
```

Instead of one CPU loop, CUDA will use **12 threads** simultaneously.

---

# Full CUDA Code

```cpp
#include <iostream>
#include <stdio.h>

__global__ void addVectors(int *A, int *B, int *C, int N)
{
    // Local thread ID inside block
    int tid = threadIdx.x;

    // Block ID inside grid
    int bid = blockIdx.x;

    // Number of threads in each block
    int blockSize = blockDim.x;

    // Global thread ID (The Golden Formula)
    int gid = bid * blockSize + tid;

    // GUARD CLAUSE: Ensure thread does not access out-of-bounds memory
    if (gid < N) 
    {
        // Perform addition
        C[gid] = A[gid] + B[gid];

        // Print thread information
        printf("Block ID: %d | Thread ID: %d | Global ID: %d | A[%d]=%d + B[%d]=%d = %d\n",
               bid, tid, gid,
               gid, A[gid],
               gid, B[gid],
               C[gid]);
    }
}

int main()
{
    const int N = 12;
    size_t bytes = N * sizeof(int);

    // Host (CPU) Arrays
    int h_A[N] = {1,2,3,4,5,6,7,8,9,10,11,12};
    int h_B[N] = {10,20,30,40,50,60,70,80,90,100,110,120};
    int h_C[N];

    // Device (GPU) Pointers
    int *d_A, *d_B, *d_C;

    // Allocate GPU memory
    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    // Copy CPU -> GPU
    cudaMemcpy(d_A, h_A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes, cudaMemcpyHostToDevice);

    // Launch kernel (3 Blocks, 4 Threads per Block)
    addVectors<<<3, 4>>>(d_A, d_B, d_C, N);

    // Wait for GPU to finish
    cudaDeviceSynchronize();

    // Copy GPU -> CPU
    cudaMemcpy(h_C, d_C, bytes, cudaMemcpyDeviceToHost);

    // Print final result
    std::cout << "\nFinal Output:\n";
    for(int i = 0; i < N; i++)
    {
        std::cout << h_C[i] << " ";
    }
    std::cout << "\n";

    // Free memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}
```

---

# Step-by-step execution

---

# Step 1: Kernel launch

This line:

```cpp
addVectors<<<3, 4>>>(d_A, d_B, d_C, N);
```

creates:

```text
Grid
 ├── Block 0
 │    ├── Thread 0
 │    ├── Thread 1
 │    ├── Thread 2
 │    └── Thread 3
 │
 ├── Block 1
 │    ├── Thread 0
 │    ├── Thread 1
 │    ├── Thread 2
 │    └── Thread 3
 │
 └── Block 2
      ├── Thread 0
      ├── Thread 1
      ├── Thread 2
      └── Thread 3
```

Total threads:

```text
3 × 4 = 12
```

---

# Step 2: Built-in variables

Inside each thread:

---

## `threadIdx.x`

Thread number inside the block.

Possible values:

```text
0,1,2,3
```

Every block starts from 0.

So:

```text
Block 0 → 0,1,2,3
Block 1 → 0,1,2,3
Block 2 → 0,1,2,3
```

---

## `blockIdx.x`

Block number:

```text
Block 0 → 0
Block 1 → 1
Block 2 → 2
```

---

## `blockDim.x`

Threads per block:

```text
4
```

Same for all blocks.

---

# Step 3: Global ID calculation

Formula:

```cpp
gid = blockIdx.x * blockDim.x + threadIdx.x;
```

Substitute values:

---

### Block 0

```text
gid = 0 * 4 + tid
```

| tid | gid |
| --- | --- |
| 0   | 0   |
| 1   | 1   |
| 2   | 2   |
| 3   | 3   |

---

### Block 1

```text
gid = 1 * 4 + tid
```

| tid | gid |
| --- | --- |
| 0   | 4   |
| 1   | 5   |
| 2   | 6   |
| 3   | 7   |

---

### Block 2

```text
gid = 2 * 4 + tid
```

| tid | gid |
| --- | --- |
| 0   | 8   |
| 1   | 9   |
| 2   | 10  |
| 3   | 11  |

---

Full view:

```text
BLOCK 0                 BLOCK 1                 BLOCK 2
[ 0 ][ 1 ][ 2 ][ 3 ]    [ 4 ][ 5 ][ 6 ][ 7 ]    [ 8 ][ 9 ][10 ][11]
```

This is exactly the global ID map. Every thread gets a 100% unique ID to find its element in the array.

---

# Step 4: Each thread does work

Inside the kernel, the thread first asks: *"Is my `gid` less than `N` (12)?"* 
Because all 12 threads pass this check, they proceed to do the work:

```cpp
C[gid] = A[gid] + B[gid];
```

So:

---

Thread gid=0:

```text
C[0] = A[0] + B[0]
     = 1 + 10
     = 11
```

---

Thread gid=1:

```text
C[1] = 2 + 20 = 22
```

---

Thread gid=2:

```text
C[2] = 3 + 30 = 33
```

---

…

---

Thread gid=11:

```text
C[11] = 12 + 120 = 132
```

---

# Actual thread execution table

| Block | Thread | Global ID | Valid? (gid < N) | Work        |
| ----- | -----: | --------: | :--------------- | ----------- |
| 0     |      0 |         0 | Yes              | A[0]+B[0]   |
| 0     |      1 |         1 | Yes              | A[1]+B[1]   |
| 0     |      2 |         2 | Yes              | A[2]+B[2]   |
| 0     |      3 |         3 | Yes              | A[3]+B[3]   |
| 1     |      0 |         4 | Yes              | A[4]+B[4]   |
| 1     |      1 |         5 | Yes              | A[5]+B[5]   |
| 1     |      2 |         6 | Yes              | A[6]+B[6]   |
| 1     |      3 |         7 | Yes              | A[7]+B[7]   |
| 2     |      0 |         8 | Yes              | A[8]+B[8]   |
| 2     |      1 |         9 | Yes              | A[9]+B[9]   |
| 2     |      2 |        10 | Yes              | A[10]+B[10] |
| 2     |      3 |        11 | Yes              | A[11]+B[11] |

---

# Expected `printf()` output

Something like:

```text
Block ID: 0 | Thread ID: 0 | Global ID: 0 | A[0]=1 + B[0]=10 = 11
Block ID: 0 | Thread ID: 1 | Global ID: 1 | A[1]=2 + B[1]=20 = 22
Block ID: 0 | Thread ID: 2 | Global ID: 2 | A[2]=3 + B[2]=30 = 33
Block ID: 0 | Thread ID: 3 | Global ID: 3 | A[3]=4 + B[3]=40 = 44

Block ID: 1 | Thread ID: 0 | Global ID: 4 | A[4]=5 + B[4]=50 = 55
Block ID: 1 | Thread ID: 1 | Global ID: 5 | A[5]=6 + B[5]=60 = 66
Block ID: 1 | Thread ID: 2 | Global ID: 6 | A[6]=7 + B[6]=70 = 77
Block ID: 1 | Thread ID: 3 | Global ID: 7 | A[7]=8 + B[7]=80 = 88

Block ID: 2 | Thread ID: 0 | Global ID: 8 | A[8]=9 + B[8]=90 = 99
Block ID: 2 | Thread ID: 1 | Global ID: 9 | A[9]=10 + B[9]=100 = 110
Block ID: 2 | Thread ID: 2 | Global ID: 10 | A[10]=11 + B[10]=110 = 121
Block ID: 2 | Thread ID: 3 | Global ID: 11 | A[11]=12 + B[11]=120 = 132
```

**Order may vary because GPU scheduling is parallel.**
Thousands of threads run at once, and they all write to the terminal simultaneously. There is no guarantee Block 0 prints before Block 1.

---

# Final result

```text
11 22 33 44 55 66 77 88 99 110 121 132
```

*(Note: What if we had 14 exam papers and launched 4 classrooms? That is 16 students total. The `if (gid < N)` guard clause tells Student 14 and Student 15 to sit quietly and do nothing, ensuring they don't grade a paper that doesn't exist!)*

---

This pattern is the absolute foundation of every CUDA program. Once this clicks, moving into dynamic blocks (`(N + threads - 1) / threads`), shared memory, warps, and matrix multiplication becomes much easier.

---

### **But the pointer `d_A` and other are created in the CPU, and lives in the RAM not in the VRAM. Why are we passing the address of the pointer to `cudaMalloc`?**

You have just stumbled upon one of the most brilliant, mind-bending concepts in C++ and CUDA: **The Pointer Paradox.** 

You are 100% correct in your observation. This is exactly what is happening:

1. **Yes, `d_A` is created by the CPU.**

2. **Yes, `d_A` lives physically in your CPU's RAM.**

3. **No, `d_A` does not live in the GPU's VRAM.**

So why are we passing the address (`&`) of a CPU RAM variable to the GPU? 

To understand this, you have to separate **"Where the pointer lives"** from **"What the pointer points to."** Let’s break it down using the **Real Estate Agent Analogy**.

---

### Concept 1: What is a pointer, really?

A pointer is just a piece of paper that holds an address. That's it. 

When you write this line inside `main()`:

```cpp
int *d_A;
```
You (the CPU) are sitting at your desk. You just placed a blank piece of paper on your desk named `d_A`. 

* Because your desk is the CPU, that piece of paper physically exists in **CPU RAM**.

* Right now, the paper is blank (or has garbage written on it). 

---

### Concept 2: Why do we pass the address (`&d_A`)?

Now, you want to buy a massive warehouse inside the GPU's VRAM. You call your Real Estate Agent, whose name is `cudaMalloc`.

You need `cudaMalloc` to buy the VRAM warehouse, and then **write the VRAM address onto your blank piece of paper**.

#### The Wrong Way: Pass by Value (No `&`)

If you wrote this:

```cpp
cudaMalloc(d_A, bytes); // Notice no &
```

In C++, if you don't use `&`, you pass by value. That means you hand the Real Estate Agent a **photocopy** of your blank piece of paper. 

1. The agent buys the GPU VRAM (let's say address `0xGPU_999`).

2. The agent writes `0xGPU_999` onto the photocopy.

3. The agent throws the photocopy in the trash and goes home. 

4. You look down at your original piece of paper (`d_A`), and **it is still blank.** Your program crashes.

#### The Right Way: Pass by Address (Using `&`)

When you write this:

```cpp
cudaMalloc(&d_A, bytes);
```

The `&` means "Address of". You are not giving the agent a photocopy. You are giving the agent **the exact GPS location of your desk in CPU RAM**, and saying: *"Here is exactly where my blank piece of paper is sitting."*

1. The agent (`cudaMalloc`) buys the GPU VRAM (address `0xGPU_999`).

2. Because the agent knows exactly where your desk is, they walk over to your desk.

3. They use an eraser, and physically write `0xGPU_999` directly onto your original `d_A` piece of paper.

4. Now, your CPU variable `d_A` successfully holds the location of the GPU VRAM!

*(Technical term: Because `d_A` is already a pointer, taking the address of it (`&d_A`) creates a **Double Pointer** (`int**`). `cudaMalloc` requires a double pointer so it can modify the original single pointer.)*

---

### Concept 3: The Split Reality

This is why CUDA memory management looks like black magic. We have a split reality. 

**The Pointer (`d_A`):**

* **Lives in:** CPU RAM (Host).

* **Role:** A street sign pointing to a destination.

**The Data (`*d_A`):**

* **Lives in:** GPU VRAM (Device).

* **Role:** The actual numbers (10, 20, 30) stored in the warehouse.

### The Ultimate Proof (Why you can't touch the data from the CPU)

Because `d_A` lives on your CPU desk, your CPU is allowed to look at the paper to see the address. But your CPU **cannot** travel to that address!

If you try to write this code in `main()`:

```cpp
int *d_A;
cudaMalloc(&d_A, bytes); // Success! d_A now holds the VRAM address.

d_A[0] = 5; // CRASH! SEGMENTATION FAULT!
```

Why does it crash? 

Your CPU looks at the paper (`d_A`), sees the address `0xGPU_999`, and tries to reach out and put the number `5` in it. But the CPU Operating System steps in and says: *"Hey! `0xGPU_999` is across the PCIe highway inside the graphics card. You don't have physical access to touch that memory from here!"*

That is exactly why you are forced to use `cudaMemcpy(..., cudaMemcpyHostToDevice)`. You have to hire a delivery truck to carry the number `5` across the PCIe highway to the VRAM address stored on your piece of paper.

<hr>

### What is a Warp? (The 32-Seat Bus)

In your code, you told the GPU to create 3 Blocks, with 4 Threads in each Block (`<<<3, 4>>>`). Logically, you think the GPU spawned exactly 12 threads. 

**Hardware Reality:** The GPU physically **cannot** spawn 4 threads. `Blocks` are divided into indivisible hardware bundles called **Warps**.

NVIDIA GPUs group threads into indivisible hardware bundles called **Warps**. 

**1 Warp = exactly 32 threads.**

Think of a Warp as a 32-seat school bus. 

* If you tell the GPU to launch a block with 32 threads, it dispatches 1 bus.

* If you tell the GPU to launch a block with 64 threads, it dispatches 2 buses.

* **If you tell the GPU to launch a block with 4 threads, it STILL dispatches a full 32-seat bus!** It just puts 4 students inside, and leaves 28 seats completely empty.

### 2. How your `<<<3, 4>>>` code actually ran on the hardware

Let's look at how much power you actually used on your RTX 3060:

You launched 3 Blocks. 

* **Block 0:** Needs 4 threads. GPU dispatches 1 Warp (32 threads).

* **Block 1:** Needs 4 threads. GPU dispatches 1 Warp (32 threads).

* **Block 2:** Needs 4 threads. GPU dispatches 1 Warp (32 threads).

**The Math:**

* Total threads you *wanted*: **12**

* Total threads the GPU physically *launched*: **96**

* Threads doing actual work: **12**

* Wasted (inactive) threads: **84**

Because you didn't launch your blocks in multiples of 32, your GPU was running at **12.5% efficiency**. 

*(This is why, in real-world code, you will always see programmers use `<<<blocks, 128>>>` or `<<<blocks, 256>>>`!)*

---

### 3. SIMT Architecture (Single Instruction, Multiple Threads)

Why does NVIDIA force threads into groups of 32? Why not just let them run independently?

Because of an architecture called **SIMT**. 

All 32 threads inside a Warp share the exact same "Manager" (Control Unit). That means **all 32 threads MUST execute the exact same line of code at the exact same nanosecond.** 

Imagine a drill sergeant yelling commands to 32 soldiers. 

* Sergeant: *"Read from array A!"* (All 32 soldiers read simultaneously).

* Sergeant: *"Read from array B!"* (All 32 soldiers read simultaneously).

* Sergeant: *"Add them together!"* (All 32 soldiers add simultaneously).

This is what makes GPUs so incredibly fast. The GPU doesn't have to fetch instructions 32 times. It fetches the instruction *once*, and applies it to 32 data points instantly.

---

### 4. Warp Divergence (The Danger of `if` statements)

This brings us to the most important part of your code: The Guard Clause.

```cpp
if (gid < N) 
{
    C[gid] = A[gid] + B[gid];
}
```

Let's look at **Block 0**. 

The GPU dispatched a Warp of 32 threads for Block 0. 

* Threads 0, 1, 2, and 3 have `gid` 0, 1, 2, and 3. 

* Threads 4 through 31 also woke up. Their `gid` is 4 through 31.

Suddenly, the Sergeant yells: *"Evaluate `if (gid < 12)`!"*

* Threads 0-3 say: **"TRUE!"**

* Threads 4-31 say: **"FALSE!"**

**The Hardware Problem:**

Remember SIMT? All 32 threads *must* execute the same instruction. They cannot split up. Thread 0 cannot do addition while Thread 31 skips ahead. 

**The Solution: Warp Divergence & Masking**

When a Warp disagrees on an `if/else` statement, the hardware experiences **Warp Divergence**. Here is exactly what the GPU does:

1. The Sergeant says: *"Okay, all FALSE threads (4-31), go to sleep (Masked off). All TRUE threads (0-3), execute the addition!"*

2. Threads 0-3 do the math: `C[gid] = A[gid] + B[gid]`. Threads 4-31 sit frozen, doing absolutely nothing, waiting for the others to finish.

3. If there was an `else` statement, the Sergeant would swap them: *"Okay, TRUE threads freeze. FALSE threads, wake up and do your part!"*

4. Once both sides of the `if/else` are finished, the paths merge, all 32 threads wake up together, and they continue down the code in lockstep.

### Summary: The Golden Rules of Warps

1. **Rule of 32:** Always design your `blockDim` (Threads per block) to be a multiple of 32 (like 32, 64, 128, 256). If you use a number like `100`, the GPU will launch 4 Warps (128 threads) and permanently waste 28 threads.

2. **Beware of Divergence:** `if` statements inside GPU code are expensive if they split a Warp in half, because the hardware literally has to run the `if` path and the `else` path sequentially while half the threads sleep. 

3. **The Guard Clause is an Exception:** Using `if (gid < N)` causes divergence on the very last Warp of your entire program. This is perfectly fine and completely unavoidable! 

By changing your launch parameters to `<<< 1, 32 >>>` (1 Block, 32 Threads), you would use exactly 1 Warp. Threads 0-11 would do the math, threads 12-31 would be safely masked off by your `if (gid < N)` guard clause, and your RTX 3060 would be perfectly happy!

<hr>
<hr>

What you have just outlined is the complete **Compilation Pipeline** of CUDA. This is one of the most brilliant engineering feats by NVIDIA, and understanding it explains exactly why CUDA dominates the high-performance computing world.

To understand this, you must first know one secret: **`nvcc` is not actually a compiler.** 

`nvcc` (NVIDIA CUDA Compiler) is a **Compiler Driver**. It is an orchestrator. When you feed it your `vadd.cu` file, `nvcc` takes out a scalpel, cuts your code perfectly in half (CPU vs. GPU), and sends each half down a completely different pipeline. 

Here is the in-depth breakdown of exactly what happens.

---

### Phase 1: The Host Code (CPU) Pipeline

Your CPU (`g++` / Ryzen processor) and your GPU (`nvcc` / RTX 3060) speak completely different languages. A standard C++ compiler like `g++` has absolutely no idea what `__global__` or `<<<1, 256>>>` means. If it sees them, it throws a syntax error.

**1. "Modified to run kernels"**
Before passing the code to your CPU compiler, `nvcc` performs a translation. It strips out all the GPU kernel code. Then, it looks at your kernel launch:
`vectorAdd<<<NUM_BLOCKS, NUM_THREADS>>>(d_a, d_b, d_c, n);`

It rewrites this weird `<<< >>>` syntax into standard, ugly C++ API functions. It transforms it into something like:
`cudaLaunchKernel((void*)vectorAdd, NUM_BLOCKS, NUM_THREADS, args);`

**2. "Compiled to x86 binary"**
Now that the file is 100% standard, pure C++, `nvcc` hands it over to your system's host compiler (in your case, `g++ 16.1`). 
`g++` compiles this code into standard **x86_64 machine code** that your AMD Ryzen 5 CPU can execute natively.

---

### Phase 2: The Device Code (GPU) Pipeline

While `g++` is busy compiling the CPU code, `nvcc` takes the `__global__` functions and starts compiling the GPU code. But it doesn't compile it to raw 1s and 0s immediately. 

**1. "Compiled to PTX"**
Instead of creating actual machine code, `nvcc` compiles your GPU code into **PTX (Parallel Thread Execution)**. 

PTX is an "Intermediate Representation." It is a fake, virtual assembly language. It acts as if it is compiling for a "perfect, theoretical" NVIDIA GPU with infinite registers and infinite memory. It is plain text, and you can actually read it if you want to!

**2. "Stable across multiple GPU generations"**
Why a fake assembly language? Because GPU hardware changes violently every 2 years. 
The physical micro-architecture of a 2016 GTX 1060 (Pascal) is wildly different from your 2021 RTX 3060 (Ampere). If NVIDIA forced you to write specific code for every single graphics card, developers would quit. 
PTX solves this. PTX is a universal, stable language. NVIDIA ensures that PTX generated 10 years ago is completely understandable by the NVIDIA driver of today.

---

### Phase 3: Execution and JIT (Just-In-Time) Compilation

So, you have your final executable file (like `./vadd`). Inside this executable is your compiled CPU code, and the text-based PTX blueprints for your GPU code. 

You press `Enter` to run the program.

**1. "PTX into native GPU instructions"**
Your physical RTX 3060 cannot read PTX. It only understands raw, physical microcode called **SASS** (Streaming ASSembler) or **cubin** (CUDA Binary). 

When your program reaches the `cudaLaunchKernel` step, the NVIDIA Display Driver on your computer wakes up. The driver looks at the PTX blueprints hidden in the executable, looks at your physical hardware, and says: *"Ah! You have an Ampere RTX 3060!"*

In a fraction of a millisecond, the driver performs **JIT (Just-In-Time) Compilation**. It translates the generic PTX text into the exact, highly-optimized, physical SASS machine code tailored perfectly for your specific GPU's silicon, and feeds it to the graphics card.

**2. "Allows for forward compatibility"**
This is the ultimate superpower of CUDA. 

Imagine you write your `vadd.cu` code today, compile it to an executable, and upload it to the internet.
Five years from now, someone downloads your executable and tries to run it on an **RTX 6090** (an architecture that hasn't even been invented yet). 

* If you had compiled it directly to RTX 3060 microcode, the program would crash. The RTX 6090 wouldn't understand the old instructions.
* **But because you embedded PTX**, the program runs perfectly. The future RTX 6090 driver will look at the PTX, JIT-compile it into RTX 6090 SASS, and run it flawlessly. 

You wrote code that runs on hardware from the future.

### Summary / How this affects your compilation flags:
Remember earlier when I told you to use `nvcc -arch=sm_86`? 
* `sm_86` is the physical architecture of your RTX 3060. 
* By using that flag, you told `nvcc`: *"Skip the JIT part for my machine! Compile it directly to physical SASS microcode for the RTX 3060 so it launches instantly!"* (This is called Ahead-Of-Time or AOT compilation).

If you were building software to sell on Steam to millions of gamers with different GPUs, you would use flags that embed the generic PTX so their individual drivers could JIT-compile it!